In [10]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

In [11]:
def larctan_basis(x, k):
    # x: (batch_size, 1, num_features)
    # k: (out_channels, num_features)
    w = k.unsqueeze(0)  # (1, out_channels, features)
    out = w * torch.atan(x)  # Elementwise
    return out  # (batch_size, out_channels, features)

In [12]:
# Load CIFAR
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)
trainset = torchvision.datasets.CIFAR10(
    root="./cifar-data", train=True, download=True, transform=transform
)
valset = torchvision.datasets.CIFAR10(
    root="./cifar-data", train=False, download=True, transform=transform
)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
valloader = DataLoader(valset, batch_size=64, shuffle=False)

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SKANLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True, basis_function=None, device='cpu'):
        super(SKANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.use_bias = bias
        self.basis_function = basis_function
        self.device = device
        if bias:
            self.weight = nn.Parameter(torch.Tensor(out_features, in_features + 1).to(device))
        else:
            self.weight = nn.Parameter(torch.Tensor(out_features, in_features).to(device))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)

    def forward(self, x):
        x = x.reshape(-1, 1, self.in_features)
        if self.use_bias:
            x = torch.cat([x, torch.ones_like(x[..., :1])], dim=2)
        y = self.basis_function(x, self.weight)
        y = torch.sum(y, dim=2)
        return y

    def extra_repr(self):
        return 'in_features={}, out_features={}'.format(
            self.in_features, self.out_features
        )

class SKANConv2d(nn.Module):
    """
    Convolutional layer using single-parameter non-linear basis functions (SKAN-style).

    Applies the basis_function over each sliding window patch and sums the responses.
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True, basis_function=None, device='cpu'):
        super(SKANConv2d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.stride = stride
        self.padding = padding
        self.use_bias = bias
        self.basis_function = basis_function
        self.device = device
        kH, kW = self.kernel_size
        self.in_features = in_channels * kH * kW
        # weight shape: (out_channels, in_features [+1 for bias])
        if bias:
            self.weight = nn.Parameter(torch.Tensor(out_channels, self.in_features + 1).to(device))
        else:
            self.weight = nn.Parameter(torch.Tensor(out_channels, self.in_features).to(device))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)

    def forward(self, x):
        # x: (batch, in_channels, H, W)
        # Extract sliding local blocks into columns
        patches = F.unfold(x, kernel_size=self.kernel_size, dilation=1, padding=self.padding, stride=self.stride)
        # patches: (batch, in_features, L) where L = H_out * W_out
        batch, in_feat, L = patches.shape
        # Optionally append bias dimension
        if self.use_bias:
            ones = torch.ones(batch, 1, L, device=patches.device)
            patches = torch.cat([patches, ones], dim=1)
        # Reshape to match SKANLinear input: (batch*L, 1, in_features(+1))
        patches = patches.permute(0, 2, 1).contiguous().reshape(-1, 1, patches.shape[1])
        # Apply basis function: returns (batch*L, out_channels, in_features(+1))
        y = self.basis_function(patches, self.weight)
        # Sum across feature dimension
        y = y.sum(dim=2)
        # Reshape back to image grid
        y = y.reshape(batch, L, self.out_channels).permute(0, 2, 1)
        H_out = (x.shape[2] + 2 * self.padding - self.kernel_size[0]) // self.stride + 1
        W_out = (x.shape[3] + 2 * self.padding - self.kernel_size[1]) // self.stride + 1
        out = y.reshape(batch, self.out_channels, H_out, W_out)
        return out

class MultSKANNetwork(nn.Module):
    def __init__(self, n_a_list, n_m_list, basis_function=None, bias=True, device='cpu'):
        """
        n_a_list: list of addition widths [n_a0, n_a1, n_a2, ...]
        n_m_list: list of multiplication widths [n_m0, n_m1, n_m2, ...]
        """
        super(MultSKANNetwork, self).__init__()
        assert len(n_a_list) == len(n_m_list), "n_a_list and n_m_list must have the same length"
        self.n_a_list = n_a_list
        self.n_m_list = n_m_list
        self.device = device

        self.layers = nn.ModuleList()
        for l in range(len(n_a_list) - 1):
            in_features = n_a_list[l] + n_m_list[l]
            out_features = n_a_list[l+1] + 2 * n_m_list[l+1]
            self.layers.append(SKANLinear(in_features, out_features, bias=bias, 
                                          basis_function=basis_function, device=device))

    def forward(self, x):
        for l, layer in enumerate(self.layers):
            x = layer(x)  # pass through SKANLinear
            n_a_next = self.n_a_list[l+1]
            n_m_next = self.n_m_list[l+1]

            if n_m_next > 0:
                # Split x into additive and multiplicative parts
                x_add = x[:, :n_a_next]  # First n_a_next elements stay the same
                x_mul_raw = x[:, n_a_next:]  # Remaining elements to be multiplied
                # Pairwise multiplication
                x_mul = x_mul_raw[:, 0::2] * x_mul_raw[:, 1::2]
                # Concatenate additive + multiplicative outputs
                x = torch.cat([x_add, x_mul], dim=1)
        return x


In [14]:
class SKAN_CIFAR10_Classifier(nn.Module):
    def __init__(self, basis_fn, device='cpu'):
        super().__init__()
        self.device = device
        # conv block 1
        self.conv1 = SKANConv2d(
            in_channels=3, out_channels=32, kernel_size=3,
            stride=1, padding=1, bias=True,
            basis_function=basis_fn, device=device
        )
        # conv block 2
        self.conv2 = SKANConv2d(
            in_channels=32, out_channels=64, kernel_size=3,
            stride=1, padding=1, bias=True,
            basis_function=basis_fn, device=device
        )
        # after two convs, CIFAR10 images 32x32 -> still 32x32
        # flatten and pass through one SKANLinear
        flattened = 64 * 32 * 32
        # one additive + multiplicative layer
        self.fc_network = MultSKANNetwork(
            n_a_list=[flattened, 128, 10],
            n_m_list=[0, 64, 0],  # using only additive SKANLinear layers here
            basis_function=basis_fn,
            bias=True,
            device=device
        )

    def forward(self, x):
        x = self.conv1(x)
        x = torch.relu(x)
        x = self.conv2(x)
        x = torch.relu(x)
        x = x.reshape(x.size(0), -1)
        x = self.fc_network(x)
        return x


def train(model, train_loader, val_loader, device, epochs=10, lr=1e-3):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    for epoch in range(1, epochs+1):
        # Training
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for inputs, targets in tqdm(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(targets).sum().item()
            total += targets.size(0)

        train_loss = running_loss / total
        train_acc = correct / total * 100

        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)
                _, preds = outputs.max(1)
                val_correct += preds.eq(targets).sum().item()
                val_total += targets.size(0)
        val_loss = val_loss / val_total
        val_acc = val_correct / val_total * 100

        print(f"Epoch {epoch:2d}: "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cifarmodel = SKAN_CIFAR10_Classifier(larctan_basis, device)
train(cifarmodel, trainloader, valloader, device, epochs=5, lr=1e-3)


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [02:25<00:00,  5.39it/s]


Epoch  1: Train Loss: 1.2394, Train Acc: 55.67% | Val Loss: 1.0237, Val Acc: 63.63%


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [02:30<00:00,  5.19it/s]


Epoch  2: Train Loss: 0.8904, Train Acc: 68.61% | Val Loss: 0.9982, Val Acc: 64.78%


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [02:30<00:00,  5.21it/s]


Epoch  3: Train Loss: 0.7074, Train Acc: 75.10% | Val Loss: 0.9488, Val Acc: 66.68%


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [02:30<00:00,  5.19it/s]


Epoch  4: Train Loss: 0.5359, Train Acc: 81.60% | Val Loss: 1.0075, Val Acc: 66.22%


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [02:27<00:00,  5.29it/s]


Epoch  5: Train Loss: 0.3731, Train Acc: 87.48% | Val Loss: 1.0690, Val Acc: 66.94%


# MNIST

In [6]:
def larctan_basis(x, k):
    w = k.unsqueeze(0)  # (1, out_channels, features)
    return w * torch.atan(x)

class ConvMultSKAN_MNIST(nn.Module):
    def __init__(self, basis_fn, device='cpu'):
        super().__init__()
        self.device = device

        self.conv1 = SKANConv2d(
            in_channels=1, out_channels=32, kernel_size=5,
            stride=1, padding=2, bias=True,
            basis_function=basis_fn, device=device
        )
        self.conv2 = SKANConv2d(
            in_channels=32, out_channels=64, kernel_size=5,
            stride=1, padding=2, bias=True,
            basis_function=basis_fn, device=device
        )
        self.pool = nn.MaxPool2d(2, 2)

        # After conv+pooling layers:
        # MNIST 28x28 -> after pool -> 14x14 -> after pool -> 7x7
        flattened = 64 * 7 * 7

        self.fc_network = MultSKANNetwork(
            n_a_list=[flattened, 128, 10],
            n_m_list=[0, 32, 0],  # second layer has multiplicative pairs
            basis_function=basis_fn,
            bias=True,
            device=device
        )

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.reshape(x.size(0), -1)
        x = self.fc_network(x)
        return x


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
val_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mnistmodel = ConvMultSKAN_MNIST(larctan_basis, device)
train(mnistmodel, train_loader, val_loader, device, epochs=10, lr=1e-3)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:53<00:00, 17.47it/s]


Epoch  1: Train Loss: 0.1295, Train Acc: 96.15% | Val Loss: 0.0391, Val Acc: 98.73%


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:54<00:00, 17.14it/s]


Epoch  2: Train Loss: 0.0424, Train Acc: 98.65% | Val Loss: 0.0349, Val Acc: 98.99%


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:53<00:00, 17.65it/s]


...

Epoch  8: Train Loss: 0.0173, Train Acc: 99.41% | Val Loss: 0.0240, Val Acc: 99.18%


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:52<00:00, 17.78it/s]


Epoch  9: Train Loss: 0.0149, Train Acc: 99.49% | Val Loss: 0.0260, Val Acc: 99.13%


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [00:52<00:00, 17.81it/s]


Epoch 10: Train Loss: 0.0153, Train Acc: 99.49% | Val Loss: 0.0301, Val Acc: 99.11%


In [15]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [19]:
mnistmodel = ConvMultSKAN_MNIST(larctan_basis, device)
cifarmodel = SKAN_CIFAR10_Classifier(larctan_basis, device)

print(f"Number of trainable parameters in MNIST Model: {count_parameters(mnistmodel):,}")
print(f"Number of trainable parameters in cifar Model: {count_parameters(cifarmodel):,}")

Number of trainable parameters in MNIST Model: 656,010
Number of trainable parameters in cifar Model: 16,798,794
